# Object Numeric-String Contract v0.2.2

**This is the final bounded contract patch before Stage 0 Data Audit.**

Không scan thêm dữ liệu, không decode media, không chạy model/GPU và không sửa source JSON.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
DATASET_ROOT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
OUTPUT_ROOT = Path('/kaggle/working/object_numeric_contract_v022')
V021_SUMMARY = Path('/kaggle/working/cross_asset_survey_v021/patch_summary_v021.json')
print('This is the final bounded contract patch before Stage 0 Data Audit.')
print('dataset:', DATASET_ROOT)
print('output:', OUTPUT_ROOT)

In [ ]:
module_file = REPO_DIR / 'src/triage_eg/data/object_numeric_contract.py'
if not module_file.is_file():
    raise RuntimeError(f'Missing {module_file}. Attach or place the repository at AIC_REPO_DIR before running this notebook.')
sys.path.insert(0, str(REPO_DIR / 'src'))
commit = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=False).stdout.strip()
print('git commit:', commit or 'UNKNOWN')
print('python:', sys.version)

In [ ]:
from triage_eg.data.object_numeric_contract import NumericLimits, run_survey, write_outputs
limits = NumericLimits(max_object_json_total=15, max_object_json_bytes=1048576)
result = run_survey(DATASET_ROOT, limits=limits, strict_root=True, v021_summary=V021_SUMMARY if V021_SUMMARY.is_file() else None)
artifact_paths = write_outputs(result, OUTPUT_ROOT)
print(result.summary['disclaimer'])

In [ ]:
coordinate_summary = result.summary['coordinate_summary']
print({k:v for k,v in coordinate_summary.items() if k not in ('valid_examples','invalid_examples')})

In [ ]:
score_summary = result.summary['score_summary']
print({k:v for k,v in score_summary.items() if k not in ('valid_examples','invalid_examples')})

In [ ]:
label_summary = result.summary['label_summary']
print({k:v for k,v in label_summary.items() if k not in ('valid_examples','invalid_examples')})

In [ ]:
print('invalid sample count:', len(result.invalid_samples))
for sample in result.invalid_samples[:20]: print(sample)

In [ ]:
print(result.summary['normalization_contract'])

In [ ]:
print(result.summary['readiness'])
print(result.summary['issues_summary'])

In [ ]:
from zipfile import ZipFile
zip_path = artifact_paths['zip']
with ZipFile(zip_path) as archive:
    members = archive.namelist()
assert len(members) == 5
assert 'object_numeric_contract_v022.zip' not in members
print('ZIP verified:', members)

In [ ]:
print('DOWNLOAD ZIP:', zip_path)
print('size_bytes:', zip_path.stat().st_size)